# Translate-then-Summarize Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Translate-then-Summarize baseline for English-to-Chinese cross-lingual dialogue summarization.

The TS pipeline uses two local small language model agents. Agent 1 reads the original English dialogue and translates the full dialogue into Chinese while preserving speaker names, turn order, placeholders, and dialogue structure. Agent 2 then reads the translated Chinese dialogue and generates a concise Chinese summary.

```text
English Dialogue
→ Agent 1: Chinese Translation Agent
→ Agent 2: Chinese Summarization Agent
→ Final Chinese Summary
```

The pipeline consists of two agents:

```text
Agent 1: Chinese Translation Agent
Input: original English dialogue
Output: translated Chinese dialogue

Agent 2: Chinese Summarization Agent
Input: translated Chinese dialogue from Agent 1
Output: final Chinese summary
```
This setup is used as a Translate-then-Summarize baseline. Unlike the Direct pipeline, TS explicitly creates an intermediate Chinese dialogue translation before summarization. This allows us to inspect whether errors come from the translation stage or the summarization stage.

The local small language model is served through Ollama. The notebook controls the prompt design, agent workflow, input/output processing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Translate-then-Summarize baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama for both agents:

- Agent 1: Chinese Translation Agent
- Agent 2: Chinese Summarization Agent

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

Make sure the model names in the notebook match the models installed in Ollama:

```bash
TRANSLATION_MODEL = "qwen3.5:27b"
SUMMARIZATION_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [ ]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# TS models
TRANSLATION_MODEL = "qwen3.5:27b"
SUMMARIZATION_MODEL = "qwen3.5:27b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for TS baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "ts_qwen27b_50samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "ts_qwen27b_50samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "ts_qwen27b_50samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples_errors.jsonl


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['qwen3.5:9b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: TS prompt templates

CHINESE_TRANSLATION_PROMPT = """You are a dialogue translation agent.

Your task is to translate the following English dialogue into Chinese.

Requirements:
- Translate the full dialogue into Chinese.
- Preserve the speaker names.
- Preserve the dialogue structure and turn order.
- Preserve placeholders such as <file_photo>, <file_other>, <location>, or emojis if they appear.
- Do not summarize the dialogue.
- Do not omit any important information.
- Do not add information that is not in the original dialogue.
- Output only the translated Chinese dialogue.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

English dialogue:
{dialogue}

Chinese translation:
"""


CHINESE_SUMMARIZATION_PROMPT = """You are a Chinese dialogue summarization agent.

Your task is to read the translated Chinese dialogue and generate a concise Chinese summary.

Requirements:
- Summarize the main information in the dialogue.
- Write the summary in Chinese.
- Keep the summary concise and faithful to the dialogue.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the final Chinese summary.

Conciseness: 
- For simple dialogues, prefer 20-50 Chinese characters. 
- For complex dialogues, allow up to 80 Chinese characters.

Chinese dialogue:
{translated_dialogue}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: TS agent functions

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def chinese_translation_agent(dialogue: str) -> str:
    """Agent 1: English dialogue -> translated Chinese dialogue."""
    prompt = fill_prompt(
        CHINESE_TRANSLATION_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()


def chinese_summarization_agent(translated_dialogue: str) -> str:
    """Agent 2: translated Chinese dialogue -> Chinese summary."""
    prompt = fill_prompt(
        CHINESE_SUMMARIZATION_PROMPT,
        {
            "translated_dialogue": translated_dialogue,
        },
    )

    response = call_ollama(
        model=SUMMARIZATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: TS pipeline

def run_ts_pipeline(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the Translate-then-Summarize pipeline."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: English dialogue -> translated Chinese dialogue
    translated_dialogue = chinese_translation_agent(dialogue)

    if verbose:
        print("\n" + "=" * 80)
        print(f"Sample ID: {sample_id}")
        print("=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===")
        print(translated_dialogue)
        print("=" * 80 + "\n")

    # Agent 2: translated Chinese dialogue -> Chinese summary
    final_chinese_summary = chinese_summarization_agent(translated_dialogue)

    if verbose:
        print("=== Agent 2 Final Output: Chinese Summary ===")
        print(final_chinese_summary)
        print("=" * 80 + "\n")

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,

        # Intermediate output from Agent 1
        "translated_dialogue": translated_dialogue,

        # Final output from Agent 2
        "final_summary": final_chinese_summary,

        # References
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        # Metadata
        "pipeline": "translate_then_summarize",
        "translation_model": TRANSLATION_MODEL,
        "summarization_model": SUMMARIZATION_MODEL,
        "num_model_calls": 2,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect whether errors come from the English summarization stage or the Chinese translation stage.

In [ ]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=50)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'gold_00001', 'test_index': 23, 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene", 'reference_english_summary': 'Mark lied to Anne about his age. Mark is 40.', 'reference_chinese_summary': '马克向安妮隐瞒了自己的年龄。他40岁了。'}


In [11]:
# Cell 10: Run the TS pipeline for the first example

result = run_ts_pipeline(test_data[4], verbose=True)
result


Sample ID: gold_00005
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
乔伊斯：快看看这个！
乔伊斯：<link>
迈克尔：太便宜了！
埃德森：不可能！我现在就订票！！

=== Agent 2 Final Output: Chinese Summary ===
乔伊斯分享了一个链接，迈克尔惊叹价格太便宜，埃德森则立即决定订票。



{'id': 'gold_00005',
 'test_index': 66,
 'dialogue': "Joyce: Check this out!\r\nJoyce: <link>\r\nMichael: That's cheap!\r\nEdson: No way! I'm booking my ticket now!! ",
 'translated_dialogue': '乔伊斯：快看看这个！\n乔伊斯：<link>\n迈克尔：太便宜了！\n埃德森：不可能！我现在就订票！！',
 'final_summary': '乔伊斯分享了一个链接，迈克尔惊叹价格太便宜，埃德森则立即决定订票。',
 'reference_english_summary': 'Edson is booking his ticket now.',
 'reference_chinese_summary': '埃德森正在订票。',
 'pipeline': 'translate_then_summarize',
 'translation_model': 'qwen3.5:27b',
 'summarization_model': 'qwen3.5:27b',
 'num_model_calls': 2}

In [12]:
# Cell 11: Print TS pipeline result clearly

def print_ts_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===")
    print(result["translated_dialogue"])
    print()

    print("=== Agent 2 Final Output: Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Translation model:", result["translation_model"])
    print("Summarization model:", result["summarization_model"])
    print("Model calls:", result["num_model_calls"])


print_ts_result(result)

=== Original Dialogue ===
Laura: Where are you?
Paul: Almost there.
Laura: Which is?
Paul: Close to the Mac.
Laura: That's so far away!
Paul: 15 mins
Laura: I am not waiting any more, see you some other time.
Paul: Please, wait!
Laura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.
Paul: I am so sorry.
Laura: I am not. 

=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
Laura: 你在哪儿？
Paul: 快到了。
Laura: 快到哪儿了？
Paul: 离麦当劳很近。
Laura: 那太远了！
Paul: 15 分钟。
Laura: 我不等了，改天再见吧。
Paul: 求你了，等等我！
Laura: 我已经等了 30 分钟了，15 分钟前你还说快到了。这太过分了。
Paul: 真的很抱歉。
Laura: 我才不觉得抱歉呢。

=== Agent 2 Final Output: Chinese Summary ===
Laura 因 Paul 迟到且未如实告知位置（谎称快到了，实则需 15 分钟）而愤怒，表示已等待 30 分钟并决定离开。尽管 Paul 道歉并请求等待，Laura 仍拒绝原谅并结束对话。

=== Reference English Summary ===
Paul is late for a meeting with Laura and she refuses to wait any longer.

=== Reference Chinese Summary ===
保罗和劳拉见面时迟到了，现在劳拉不想再等了。

=== Metadata ===
Pipeline: translate_then_summarize
Translation model: qwen3.5

## 4. Save Results

This saves the intermediate English summary and the final Chinese summary.

In [13]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed ST result to the JSONL output file.

If the notebook stops, already processed examples remain saved.

In [ ]:
# Cell 13: Batch inference with TS pipeline
# Time stamp: 2m 51.7s

MAX_EXAMPLES = 50
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running TS pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        print(f"Skipping already processed sample: {sample_id}")
        continue

    try:
        record = run_ts_pipeline(ex, verbose=True)

        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running TS pipeline:   0%|          | 0/5 [00:00<?, ?it/s]


Sample ID: gold_00001
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
Laura: 你在哪儿？
Paul: 快到了。
Laura: 哪儿？
Paul: 离麦当劳很近。
Laura: 那太远了！
Paul: 15 分钟。
Laura: 我不等了，改天再见。
Paul: 求你了，等等！
Laura: 我已经等了 30 分钟了，15 分钟前你还说快到了。这太过分了。
Paul: 真的很抱歉。
Laura: 我才不呢。

=== Agent 2 Final Output: Chinese Summary ===
Laura 因 Paul 迟到且未按时到达而生气。Paul 称离麦当劳很近，只需 15 分钟，但 Laura 表示已等待 30 分钟，且 15 分钟前 Paul 就声称快到了，因此决定不再等待并取消见面。尽管 Paul 道歉并请求等待，Laura 仍拒绝并离开。


Sample ID: gold_00002
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
Finn: 嘿  
Zadie: 嗨！最近怎么样？  
Finn: 一切都好。你呢？  
Zadie: 还不错，谢谢。  
Finn: 听着，我打算明天去一个叫象堡（Elephant and Castle）的街区，听说那里到处都是拉丁美洲的东西。有兴趣一起去吗？  
Zadie: 当然！不过你说的“东西”具体指什么？😂  
Finn: 哈哈，据说那里是些来自“拉丁美洲”（具体是哪几个国家谁也不知道）的普通人开始经营小生意和餐馆，逐渐形成了一个温馨的小社区。  
Zadie: 哦，真酷！  
Finn: 后来资本主义来了，那里很快就要被拆掉了，所以这算是最后的机会了。  
Zadie: 太可惜了 :( 是啊，我好久没吃过拉丁美洲 😂 美食了，所以完全没问题。  
Finn: 迫不及待想尝尝这种“来源不明”的拉丁风味了，哈哈。  
Zadie: 😂😂😂  
Finn: 不过，如果你愿意的话，我们可以具体约定时间和地点。  
Zadie: 我可能有点心动了，哈哈。我觉得傍晚早些时候，大概两点怎么样？  
Finn: 行，我没问题。我们

## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.

For the ST pipeline, the CSV also includes the intermediate English summary from Agent 1.

In [15]:
# Cell 14: Export TS summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),

        # Intermediate output from Agent 1
        "translated_dialogue": record.get("translated_dialogue", ""),

        # Final output from Agent 2
        "final_summary": record.get("final_summary", ""),

        # References
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        # Metadata
        "pipeline": record.get("pipeline", "translate_then_summarize"),
        "translation_model": record.get("translation_model", ""),
        "summarization_model": record.get("summarization_model", ""),
        "num_model_calls": record.get("num_model_calls", 2),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_5samples.csv


,id,test_index,dialogue,translated_dialogue,final_summary,reference_english_summary,reference_chinese_summary,pipeline,translation_model,summarization_model,num_model_calls
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,Laura: 你在哪儿？\nPaul: 快到了。\nLaura: 哪儿？\nPaul: 离麦...,Laura 因 Paul 迟到且未按时到达而生气。Paul 称离麦当劳很近，只需 15 分钟...,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn: 嘿 \nZadie: 嗨！最近怎么样？ \nFinn: 一切都好。你呢？ ...,Finn 邀请 Zadie 明天去即将被拆除的象堡（Elephant and Castle）...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,Josh: 我需要买一台 iPad 吗？\nJosh: 你觉得苹果是个好选择吗？\nBria...,Josh 咨询购买平板电脑的建议，Brian 认为苹果性价比不高，推荐了三星、小米和索尼等品...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank: 你在干嘛？？\nAndy: 看《绿箭侠》B)\nFrank: 你明天不是有小测...,Frank 提醒 Andy 明天有小测验，劝他停止看《绿箭侠》并立即复习。Andy 认为测验...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,Crystal: <file_photo>\nIrene: 他好大啊！\nCrystal: ...,Crystal 分享孩子长得太快、衣服穿不下的照片，Irene 提议带侄子去购物并承诺买单。...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2


In [16]:
# Cell 15: Compare TS outputs with references

comparison_columns = [
    "id",
    "test_index",
    "translated_dialogue",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,translated_dialogue,final_summary,reference_chinese_summary
0,gold_00001,59,Laura: 你在哪儿？\nPaul: 快到了。\nLaura: 哪儿？\nPaul: 离麦...,Laura 因 Paul 迟到且未按时到达而生气。Paul 称离麦当劳很近，只需 15 分钟...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn: 嘿 \nZadie: 嗨！最近怎么样？ \nFinn: 一切都好。你呢？ ...,Finn 邀请 Zadie 明天去即将被拆除的象堡（Elephant and Castle）...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh: 我需要买一台 iPad 吗？\nJosh: 你觉得苹果是个好选择吗？\nBria...,Josh 咨询购买平板电脑的建议，Brian 认为苹果性价比不高，推荐了三星、小米和索尼等品...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank: 你在干嘛？？\nAndy: 看《绿箭侠》B)\nFrank: 你明天不是有小测...,Frank 提醒 Andy 明天有小测验，劝他停止看《绿箭侠》并立即复习。Andy 认为测验...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal: <file_photo>\nIrene: 他好大啊！\nCrystal: ...,Crystal 分享孩子长得太快、衣服穿不下的照片，Irene 提议带侄子去购物并承诺买单。...,艾琳会带克里斯特尔的儿子去买衣服。


In [17]:
# Cell 16: Inspect TS outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "translated_dialogue",
        "final_summary",
        "reference_english_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,translated_dialogue,final_summary,reference_english_summary,reference_chinese_summary
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,Laura: 你在哪儿？\nPaul: 快到了。\nLaura: 哪儿？\nPaul: 离麦...,Laura 因 Paul 迟到且未按时到达而生气。Paul 称离麦当劳很近，只需 15 分钟...,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn: 嘿 \nZadie: 嗨！最近怎么样？ \nFinn: 一切都好。你呢？ ...,Finn 邀请 Zadie 明天去即将被拆除的象堡（Elephant and Castle）...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,Josh: 我需要买一台 iPad 吗？\nJosh: 你觉得苹果是个好选择吗？\nBria...,Josh 咨询购买平板电脑的建议，Brian 认为苹果性价比不高，推荐了三星、小米和索尼等品...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank: 你在干嘛？？\nAndy: 看《绿箭侠》B)\nFrank: 你明天不是有小测...,Frank 提醒 Andy 明天有小测验，劝他停止看《绿箭侠》并立即复习。Andy 认为测验...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,Crystal: <file_photo>\nIrene: 他好大啊！\nCrystal: ...,Crystal 分享孩子长得太快、衣服穿不下的照片，Irene 提议带侄子去购物并承诺买单。...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。
